# 🎙️ LifeMate Voice-to-Action Agentic AI — Kaggle Inference Runner
### Zalo AI Challenge 2026

This notebook runs the complete offline Voice-to-Action pipeline on Kaggle GPU (Tesla T4 / P100) to generate the official `submission.jsonl`.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip install -q soundfile accelerate

In [ ]:
# 3. Clone private repository from GitHub
# Since the repo is private, you need a Personal Access Token (PAT) with 'repo' read scope.
# Create one here: https://github.com/settings/tokens (classic token -> check 'repo')
import os

# Option A: Via Kaggle Secrets (Recommended: Add 'GITHUB_PAT' in Add-ons -> Secrets)
GITHUB_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_PAT")
except Exception:
    pass

# Option B: Or directly paste token below if not using Kaggle Secrets:
if not GITHUB_TOKEN:
    GITHUB_TOKEN = "ghp_YOUR_TOKEN_HERE"  # <-- Replace with your GitHub Token

REPO_NAME = "Quandinh030106/LifeMate-Voice2Action-Agent"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_NAME}.git"

if not os.path.exists("/kaggle/working/LifeMate-Voice2Action-Agent"):
    !git clone $REPO_URL /kaggle/working/LifeMate-Voice2Action-Agent

%cd /kaggle/working/LifeMate-Voice2Action-Agent

In [ ]:
# 4. Locate Dataset Directory
# Point this to your uploaded Kaggle Dataset containing train+public_test
import glob
datasets = glob.glob("/kaggle/input/**/public_test.jsonl", recursive=True)
if datasets:
    DATA_DIR = os.path.dirname(os.path.dirname(datasets[0]))
    print("Found Data Directory:", DATA_DIR)
else:
    DATA_DIR = "/kaggle/input/lifemate-data/train+public_test/train+public_test"
    print("Defaulting to:", DATA_DIR)

In [ ]:
# 5. Run Full Inference on public_test (Voice -> ASR -> Agent -> zaic_tools -> submission.jsonl)
!python inference.py \
    --data_dir "$DATA_DIR" \
    --split public_test \
    --output /kaggle/working/submission.jsonl

In [ ]:
# 6. Validate Output Schema with official zaic_tools validator
!python -m zaic_tools \
    --split public_test \
    --root "$DATA_DIR" \
    --validate /kaggle/working/submission.jsonl

In [ ]:
# 7. Quick check line count
!wc -l /kaggle/working/submission.jsonl
!head -n 3 /kaggle/working/submission.jsonl